# Amazon Search Ranking — Week 8 V2
## Final Ablation + Model Progression Analysis

**Upload only the Week 8 V1 result ZIP.**

This notebook creates report-ready progression, ablation, cold-start, retrieval trade-off tables, figures, and narrative.

In [ ]:

# ============================================================
# AMAZON SEARCH RANKING — WEEK 8 V2
# FINAL ABLATION + MODEL PROGRESSION ANALYSIS (ONE CLICK)
#
# Upload ONLY:
#   amazon_week8_v1_final_benchmark_results.zip
#
# Output:
#   amazon_week8_v2_ablation_progression_results.zip
# ============================================================

import os, zipfile, shutil, glob
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

try:
    from google.colab import files
    uploaded = files.upload()
    candidates = [os.path.abspath(n) for n in uploaded.keys()
                  if "week8_v1_final_benchmark_results" in n.lower() and n.lower().endswith(".zip")]
except Exception:
    candidates = glob.glob("*week8_v1_final_benchmark_results*.zip")

if not candidates:
    raise FileNotFoundError("Please upload amazon_week8_v1_final_benchmark_results.zip")

V1 = candidates[0]
OUT = Path("/content/week8_v2_ablation_progression_results")
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True)

def read_csv(member):
    with zipfile.ZipFile(V1) as z:
        with z.open(member) as f:
            return pd.read_csv(f)

ranking = read_csv("week8_v1_ranking_benchmark.csv")
ablation = read_csv("week8_v1_ablation_results.csv")
retrieval = read_csv("week8_v1_retrieval_benchmark.csv")
cold = read_csv("week8_v1_coldstart_benchmark.csv")
lat = read_csv("week8_v1_stage_latency.csv")
sysm = read_csv("week8_v1_system_metrics.csv")

# 1) Comparable model progression
order = ["BM25","SBERT","XGBoost Pointwise","LambdaMART"]
prog = ranking[ranking["model"].isin(order)].copy()
prog["step"] = prog["model"].map({m:i+1 for i,m in enumerate(order)})
prog = prog.sort_values("step").reset_index(drop=True)
prog["delta_vs_previous_abs"] = prog["NDCG@10"].diff()
prog["delta_vs_previous_pct"] = prog["NDCG@10"].pct_change()*100
bm25 = float(prog.loc[prog["model"].eq("BM25"),"NDCG@10"].iloc[0])
prog["gain_vs_BM25_abs"] = prog["NDCG@10"] - bm25
prog["gain_vs_BM25_pct"] = (prog["NDCG@10"]/bm25 - 1)*100
prog.to_csv(OUT/"week8_v2_model_progression.csv", index=False)

# 2) Ablation contribution
full = float(ablation.loc[ablation["ablation"].eq("Full Model"),"NDCG@10"].iloc[0])
abl = ablation[~ablation["ablation"].eq("Full Model")].copy()
abl["feature_group"] = abl["ablation"].str.replace("Without ","",regex=False)
abl["absolute_drop"] = full - abl["NDCG@10"]
abl["relative_drop_pct"] = abl["absolute_drop"]/full*100
abl["importance_rank"] = abl["absolute_drop"].rank(method="dense",ascending=False).astype(int)
abl = abl.sort_values("absolute_drop",ascending=False)
abl[["importance_rank","feature_group","NDCG@10","absolute_drop","relative_drop_pct",
     "Recall@10","MRR"]].to_csv(OUT/"week8_v2_ablation_contribution.csv",index=False)

# 3) Cold-start comparison
keep = ["BM25","SBERT","LLM-only","ZeroShot+LLM","LambdaMART"]
cold_sel = cold[cold["model"].isin(keep)].copy().sort_values("NDCG@10",ascending=False)
lm = float(cold.loc[cold["model"].eq("LambdaMART"),"NDCG@10"].iloc[0])
zs = float(cold.loc[cold["model"].eq("ZeroShot+LLM"),"NDCG@10"].iloc[0])
cold_sel["gap_to_LambdaMART_abs"] = lm - cold_sel["NDCG@10"]
cold_sel["pct_of_LambdaMART_NDCG"] = cold_sel["NDCG@10"]/lm*100
cold_sel.to_csv(OUT/"week8_v2_coldstart_comparison.csv",index=False)

# 4) Retrieval trade-off
ret = retrieval.copy()
ret["ndcg_rank"] = ret["NDCG@10"].rank(method="min",ascending=False).astype(int)
ret["recall50_rank"] = ret["Recall@50"].rank(method="min",ascending=False).astype(int)
ret["balanced_rank_sum"] = ret["ndcg_rank"] + ret["recall50_rank"]
ret.sort_values(["balanced_rank_sum","NDCG@10"],ascending=[True,False]).to_csv(
    OUT/"week8_v2_retrieval_tradeoff.csv",index=False)

# 5) Key findings
best = ranking.sort_values("NDCG@10",ascending=False).iloc[0]
best_map = ranking.sort_values("MAP@10",ascending=False).iloc[0]
def drop(name):
    return float(abl.loc[abl["feature_group"].eq(name),"relative_drop_pct"].iloc[0])

meta_drop = drop("Metadata_Structural")
lex_drop = drop("Lexical")
sem_drop = drop("Semantic_SBERT")
clip_drop = drop("Multimodal_CLIP")
fresh = float(sysm.loc[sysm["metric"].eq("V1.3 fresh mean latency (ms)"),"value"].iloc[0])

findings = pd.DataFrame([
    ["Final ranking winner",f"{best['model']} achieved NDCG@10={best['NDCG@10']:.4f} and MRR={best['MRR']:.4f}."],
    ["Gain over BM25",f"LambdaMART improved NDCG@10 by {(best['NDCG@10']/bm25-1)*100:.2f}% relative to BM25."],
    ["MAP nuance",f"{best_map['model']} had the highest MAP@10 ({best_map['MAP@10']:.4f}), while LambdaMART led NDCG@10/MRR."],
    ["Largest ablation impact",f"Removing metadata/structural features reduced NDCG@10 by {meta_drop:.2f}%."],
    ["Lexical importance",f"Removing lexical features reduced NDCG@10 by {lex_drop:.2f}%."],
    ["Semantic importance",f"Removing SBERT features reduced NDCG@10 by {sem_drop:.2f}%."],
    ["Multimodal importance",f"Removing CLIP reduced NDCG@10 by {clip_drop:.2f}%."],
    ["Cold-start",f"ZeroShot+LLM achieved {zs:.4f} vs LambdaMART {lm:.4f}; gap={lm-zs:.4f}."],
    ["Serving",f"Optimized fresh mean latency was {fresh:.2f} ms (<200 ms target)."],
],columns=["finding","interpretation"])
findings.to_csv(OUT/"week8_v2_key_findings.csv",index=False)

# 6) Plots
plt.figure(figsize=(9,5))
plt.plot(prog["model"],prog["NDCG@10"],marker="o")
for x,y in zip(prog["model"],prog["NDCG@10"]):
    plt.text(x,y+0.004,f"{y:.3f}",ha="center",fontsize=9)
plt.ylabel("NDCG@10")
plt.title("Week 8 V2 — Model Progression")
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig(OUT/"week8_v2_model_progression.png",dpi=180)
plt.close()

plt.figure(figsize=(9,5))
d=abl.sort_values("relative_drop_pct",ascending=True)
plt.barh(d["feature_group"],d["relative_drop_pct"])
plt.xlabel("NDCG@10 drop when removed (%)")
plt.title("Week 8 V2 — Feature Group Ablation Contribution")
plt.tight_layout()
plt.savefig(OUT/"week8_v2_ablation_contribution.png",dpi=180)
plt.close()

plt.figure(figsize=(9,5))
d=cold_sel.sort_values("NDCG@10",ascending=True)
plt.barh(d["model"],d["NDCG@10"])
plt.xlabel("NDCG@10")
plt.title("Week 8 V2 — Cold-Start Holdout Comparison")
plt.tight_layout()
plt.savefig(OUT/"week8_v2_coldstart_comparison.png",dpi=180)
plt.close()

plt.figure(figsize=(7,5))
plt.scatter(retrieval["Recall@50"],retrieval["NDCG@10"],s=80)
for _,r in retrieval.iterrows():
    plt.annotate(r["model"],(r["Recall@50"],r["NDCG@10"]),
                 xytext=(5,5),textcoords="offset points")
plt.xlabel("Recall@50")
plt.ylabel("NDCG@10")
plt.title("Week 8 V2 — Retrieval Quality vs Coverage")
plt.tight_layout()
plt.savefig(OUT/"week8_v2_retrieval_tradeoff.png",dpi=180)
plt.close()

# 7) Report-ready narrative
report=f"""# Week 8 V2 — Final Ablation & Model Progression Analysis

## Main progression
BM25 NDCG@10 = {bm25:.4f}. LambdaMART NDCG@10 = {float(prog.loc[prog['model'].eq('LambdaMART'),'NDCG@10'].iloc[0]):.4f}.
The final LambdaMART model improves NDCG@10 by {(float(prog.loc[prog['model'].eq('LambdaMART'),'NDCG@10'].iloc[0])/bm25-1)*100:.2f}% relative to BM25.

The largest jump comes from supervised learning-to-rank rather than from representation replacement alone.
This suggests the system benefits most from learning how to combine lexical, semantic, multimodal, and structural evidence.

## Ablation
Removing metadata/structural features produces the largest NDCG@10 decrease ({meta_drop:.2f}%).
Removing lexical features decreases NDCG@10 by {lex_drop:.2f}%.
Removing SBERT decreases NDCG@10 by {sem_drop:.2f}%.
Removing CLIP decreases NDCG@10 by {clip_drop:.2f}%.

## Cold-start
ZeroShot+LLM NDCG@10 = {zs:.4f}; LambdaMART = {lm:.4f}; absolute gap = {lm-zs:.4f}.
This shows the zero-shot cold-start strategy recovers most of the supervised reference quality.

## Retrieval trade-off
Full-catalog retrieval should be reported separately from reranking.
Semantic/multimodal retrieval improves judged ranking quality, while BM25 retains strong Recall@50 coverage.

## Deployment
Optimized fresh end-to-end latency = {fresh:.2f} ms, below the 200 ms target.

## Reporting rule
Keep reranking, full-catalog retrieval, and cold-start results in separate evaluation panels because they use different scopes/splits.
"""
(OUT/"week8_v2_report_ready_analysis.md").write_text(report,encoding="utf-8")

pd.DataFrame([{"file":p.name,"purpose":"Week 8 V2 analysis output"}
              for p in sorted(OUT.iterdir())]).to_csv(OUT/"week8_v2_manifest.csv",index=False)

ZIP="/content/amazon_week8_v2_ablation_progression_results.zip"
if os.path.exists(ZIP):
    os.remove(ZIP)
with zipfile.ZipFile(ZIP,"w",zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):
        z.write(p,arcname=p.name)

print("=== WEEK 8 V2 COMPLETE ===")
print("\nMODEL PROGRESSION")
print(prog[["model","NDCG@10","delta_vs_previous_pct","gain_vs_BM25_pct"]].to_string(index=False))
print("\nABLATION CONTRIBUTION")
print(abl[["feature_group","relative_drop_pct"]].to_string(index=False))
print("\nOutput:",ZIP)

try:
    files.download(ZIP)
except Exception:
    pass
